In [ ]:
from datasets import load_dataset
from datasets import train

Скачаем датасет по этой ссылке https://www.kaggle.com/datasets/Cornell-University/arxiv и преобразуем его в hf dataset

In [12]:
ds = load_dataset("json", data_files="arxiv-metadata-oai-snapshot.json")

In [9]:
tags_to_names = {
    "cs": "Computer Science",
    "econ": "Economics",
    "eess": "Electrical Engineering and Systems Science",
    "math": "Mathematics",
    "physics": "Physics",
    "q-bio": "Quantitive Biology",
    "q-fin": "Quantitive Finance",
    "stat": "Statistics"
}

In [10]:
tags_to_labels = {tag : label for tag,label in zip(tags_to_names, range(len(tags_to_names)))}
print(tags_to_labels)

{'cs': 0, 'econ': 1, 'eess': 2, 'math': 3, 'physics': 4, 'q-bio': 5, 'q-fin': 6, 'stat': 7}


In [14]:
def add_label(example):

    tags = example['categories'].split() 
    
    example['label'] = tags_to_labels.get(tags[0].split('.')[0], 4)
    return example

ds = ds.map(add_label, keep_in_memory=True)

Map:   0%|          | 0/3002164 [00:00<?, ? examples/s]

In [15]:
def add_soft_label(example):
    num_classes = len(tags_to_labels)
    soft = [0.0] * num_classes
    
    tags = example['categories'].split() 
    prefixes = list(set(t.split('.')[0] for t in tags))
    
    mapped_labels = [tags_to_labels.get(p, 4) for p in prefixes]
    unique_mapped = list(set(mapped_labels))
    
    weight = 1.0 / len(unique_mapped)
    for idx in unique_mapped:
        soft[idx] = weight
    
    example['soft_labels'] = soft
    return example

ds = ds.map(add_soft_label, keep_in_memory=True)

Map:   0%|          | 0/3002164 [00:00<?, ? examples/s]

In [16]:
def count_frequencies(ds):
  class_frequencies = len(tags_to_names) * [0]
  for elem in ds:
    class_frequencies[elem['label']] += 1
  return class_frequencies

In [19]:
class_frequencies = count_frequencies(ds['train'])

In [20]:
class_frequencies

[742883, 11062, 71120, 573194, 1495760, 33805, 13327, 61013]

Сбалансируем датасет оставив по 25000 примеров с каждого класса

In [21]:
from datasets import concatenate_datasets

num_classes = len(tags_to_labels)

MAX_PER_CLASS = 25000

balanced = []
for label in range(num_classes):
    subset = ds["train"].filter(lambda x, l=label: x['label'] == l)
    if len(subset) > MAX_PER_CLASS:
        subset = subset.shuffle(seed=42).select(range(MAX_PER_CLASS))
    balanced.append(subset)

ds_balanced = concatenate_datasets(balanced).shuffle(seed=42)
print(f"Размер после балансировки: {len(ds_balanced)}")

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3002164 [00:00<?, ? examples/s]

Размер после балансировки: 174389


In [23]:
class_frequencies = count_frequencies(ds_balanced)

In [24]:
class_frequencies

[25000, 11062, 25000, 25000, 25000, 25000, 13327, 25000]

оставим только нужные колонки

In [25]:
keep_columns = ['title', 'abstract', 'label', 'soft_labels']
ds_balanced= ds_balanced.remove_columns([c for c in ds_balanced.column_names if c not in keep_columns])

In [ ]:
from huggingface_hub import login

login("")

In [34]:
ds['train'] = ds_balanced

In [39]:
ds = ds.class_encode_column("label")

Stringifying the column:   0%|          | 0/174389 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/174389 [00:00<?, ? examples/s]

In [40]:
ds = ds['train'].train_test_split(test_size=0.2, stratify_by_column='label')

In [42]:
ds.push_to_hub("pinmax/arxiv_balanced_soft_labels")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/pinmax/arxiv_balanced_soft_labels/commit/691269d5b19696b21d2c414b6fa7138a3d4024d6', commit_message='Upload dataset', commit_description='', oid='691269d5b19696b21d2c414b6fa7138a3d4024d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pinmax/arxiv_balanced_soft_labels', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pinmax/arxiv_balanced_soft_labels'), pr_revision=None, pr_num=None)